In [1]:
import os
from pathlib import Path
BASE_DIR = Path.cwd()
ROOT_DIR = BASE_DIR.parent
CACHE_DIR = str(ROOT_DIR.parent / "huggingface_cache")

os.environ["HF_HOME"] = CACHE_DIR
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import random
import json
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()

DATA_PRETRAIN_DIR = (ROOT_DIR / "data/pretrain_data/processed").resolve()
DATA_SFT_DIR = (ROOT_DIR / "data/sft_data/qna.json").resolve()

DATA_DIR = (ROOT_DIR / "data/eval_data/tele-qna.json").resolve()
DATA_OQNA_DIR = (ROOT_DIR / "data/eval_data/tele-eval-10k.json").resolve()

IMAGES_DIR = (ROOT_DIR / "images").resolve()
LOG_PRETRAIN_DIR = (ROOT_DIR / "notebooks/wandb/run-20260307_222127-qcn1cesx/files/output.log").resolve()
LOG_SFT_DIR = (ROOT_DIR / "notebooks/wandb/run-20260326_000428-3o09knd7/files/output.log").resolve()

RESULT_MCQ_OR_DIR = (ROOT_DIR / "data/benchmark_results/tele-qna-with-qwen3-original.json").resolve()
RESULT_OQNA_OR_DIR = (ROOT_DIR / "data/benchmark_results/tele-eval-with-qwen3-original.json").resolve()
 
RESULT_MCQ_DIR = (ROOT_DIR / "data/benchmark_results/tele-qna-with-qwen3-tele_2376.json").resolve()
RESULT_OQNA_DIR = (ROOT_DIR / "data/benchmark_results/tele-eval-with-qwen3-tele_2376.json").resolve()

MODEL_DIR = (ROOT_DIR / "models").resolve()

In [1]:
import torch
print(torch.version.cuda)

13.0


In [3]:
def get_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(
        "unsloth/Qwen3-8B",
        trust_remote_code=True
    )
    tokenizer = get_chat_template(tokenizer, chat_template="qwen-3")

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    return tokenizer

In [4]:
import os
import json
from tqdm import tqdm

def count_tokens_in_file(file_path, tokenizer, fields=("text",)):
    total_tokens = 0
    total_samples = 0

    def process_item(item):
        nonlocal total_tokens, total_samples

        texts = []
        for field in fields:
            val = item.get(field, "")
            if isinstance(val, str):
                texts.append(val)
            elif isinstance(val, list):
                texts.extend([str(x) for x in val])
            else:
                texts.append(str(val))

        if texts:
            merged_text = " ".join(texts)

            tokens = tokenizer(
                merged_text,
                add_special_tokens=False
            )["input_ids"]

            total_tokens += len(tokens)
            total_samples += 1

    with open(file_path, "r", encoding="utf-8") as f:
        try:
            data = json.load(f)

            if isinstance(data, list):
                iterable = data
            elif isinstance(data, dict):
                iterable = [data]
            else:
                raise ValueError("Unsupported JSON format")

            for item in tqdm(iterable, desc=os.path.basename(file_path)):
                process_item(item)

        except json.JSONDecodeError:
            f.seek(0)
            for line in tqdm(f, desc=os.path.basename(file_path)):
                try:
                    item = json.loads(line)
                    process_item(item)
                except Exception as e:
                    print(f"Error line in {file_path}: {e}")

    return total_tokens, total_samples

In [5]:
def count_tokens_in_folder(folder_path, tokenizer):
    total_tokens = 0
    total_samples = 0

    for file_name in os.listdir(folder_path):
        if not file_name.endswith(".jsonl"):
            continue

        file_path = os.path.join(folder_path, file_name)

        tokens, samples = count_tokens_in_file(file_path, tokenizer)
        total_tokens += tokens
        total_samples += samples

    return total_tokens, total_samples

In [6]:
tokenizer = get_tokenizer()

Model does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


# Pretrain DATA

In [8]:
import os
import json
from collections import defaultdict

with open("mapping.json") as f:
    mapping = json.load(f)

spec2group = {}
for group, specs in mapping.items():
    for s in specs:
        base = s.split("-")[0]
        spec2group[base] = group


def normalize_filename_to_spec(filename: str):
    name = filename.split(".")[0]
    name = name.split("-")[0]

    if len(name) < 5:
        return None

    return f"{name[:2]}.{name[2:5]}"


def count_tokens_by_group(folder_path, tokenizer):
    group_tokens = defaultdict(int)
    group_samples = defaultdict(int)

    for file in os.listdir(folder_path):
        if not file.endswith(".jsonl"):
            continue

        file_path = os.path.join(folder_path, file)

        with open(file_path) as f:
            for line in f:
                data = json.loads(line)

                filename = data.get("file_name")
                text = data.get("text", "")

                if not filename or not text:
                    continue

                spec = normalize_filename_to_spec(filename)
                group = spec2group.get(spec, "UNKNOWN")

                tokens = len(tokenizer.encode(text))

                group_tokens[group] += tokens
                group_samples[group] += 1

    return group_tokens, group_samples


for folder in ["3GPP"]:
    path = os.path.join(DATA_PRETRAIN_DIR, folder)

    group_tokens, group_samples = count_tokens_by_group(path, tokenizer)

    print(f"\n===== Folder {folder} =====")

    total_tokens = sum(group_tokens.values())
    total_samples = sum(group_samples.values())

    print(f"Total samples: {total_samples}")
    print(f"Total tokens: {total_tokens}")

    for g in sorted(group_tokens.keys()):
        t = group_tokens[g]
        s = group_samples[g]
        avg = t / s if s else 0

        print(f"{g}: samples={s}, tokens={t}, avg={avg:.2f}")


===== Folder 3GPP =====
Total samples: 45892
Total tokens: 33444966
CT1: samples=3094, tokens=2846357, avg=919.96
CT3: samples=2180, tokens=2502892, avg=1148.12
CT4: samples=6291, tokens=4600400, avg=731.27
CT6: samples=647, tokens=288535, avg=445.96
RAN1: samples=1854, tokens=1833758, avg=989.08
RAN2: samples=2135, tokens=1313179, avg=615.07
RAN3: samples=1691, tokens=1985560, avg=1174.19
RAN4: samples=1660, tokens=614292, avg=370.06
RAN5: samples=593, tokens=436381, avg=735.89
SA1: samples=2538, tokens=849144, avg=334.57
SA2: samples=4442, tokens=5791360, avg=1303.77
SA3: samples=3487, tokens=1842692, avg=528.45
SA4: samples=2492, tokens=1175382, avg=471.66
SA5: samples=3830, tokens=1792666, avg=468.06
SA6: samples=1256, tokens=748927, avg=596.28
UNKNOWN: samples=7702, tokens=4823441, avg=626.26


# SFT DATA

In [17]:
tokens, samples = count_tokens_in_file(DATA_SFT_DIR, tokenizer, ('instruction', 'output'))

print(f"File: {DATA_SFT_DIR}")
print(f"Samples: {samples}")
print(f"Total tokens: {tokens}")
print(f"Avg tokens/sample: {tokens / samples if samples else 0:.2f}")

qna.json: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20000/20000 [00:14<00:00, 1339.05it/s]

File: /mnt/PACS/Data/thinhnh/phamngocdo/AI-Agent-System-in-Telecom/data/sft_data/qna.json
Samples: 20000
Total tokens: 4807095
Avg tokens/sample: 240.35


In [18]:
total = 20000
tele_mcq = 4100
general_instr = 3000
tele_instr = total - tele_mcq - general_instr

def pct(x):
    return x / total * 100

print("=== Dataset Statistics ===")
print(f"Total samples: {total}\n")

print(f"Tele-MCQ: {tele_mcq} ({pct(tele_mcq):.2f}%)")
print(f"General Instruction: {general_instr} ({pct(general_instr):.2f}%)")
print(f"Tele-Instruction: {tele_instr} ({pct(tele_instr):.2f}%)")

=== Dataset Statistics ===
Total samples: 20000

Tele-MCQ: 4100 (20.50%)
General Instruction: 3000 (15.00%)
Tele-Instruction: 12900 (64.50%)


In [22]:
def count_empty_think(file_path):
    empty_count = 0
    total = 0

    pattern = re.compile(r"^\s*<think>(.*?)</think>", re.DOTALL)

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for item in tqdm(data):
        output = item.get("output", "")
        total += 1

        match = pattern.match(output)

        if match:
            content = match.group(1).strip()
            if content == "":
                empty_count += 1
        else:
            empty_count += 1

    print(f"Total samples: {total}")
    print(f"Have <think>: {total - empty_count}")
    print(f"Percent: { (total- empty_count) / total * 100:.2f}%")

count_empty_think(DATA_SFT_DIR)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20000/20000 [00:00<00:00, 96235.72it/s]

Total samples: 20000
Have <think>: 5098
Percent: 25.49%


# RESULT

### TRAINING

In [35]:
def parse_log_file(log_path):
    with open(log_path, "r", encoding="utf-8") as f:
        text = f.read()

    matches = re.findall(r"\{.*?\}", text)

    train_epochs, train_loss = [], []
    eval_epochs, eval_loss = [], []

    for m in matches:
        try:
            d = eval(m)

            if "loss" in d:
                train_epochs.append(d.get("epoch"))
                train_loss.append(d["loss"])

            if "eval_loss" in d:
                eval_epochs.append(d.get("epoch"))
                eval_loss.append(d["eval_loss"])

        except:
            continue

    return train_epochs, train_loss, eval_epochs, eval_loss

In [52]:
def plot_single_log(log_path, title, save_path, train_step=10, eval_step=300):
    te, tl, ee, el = parse_log_file(log_path)

    ratio = eval_step / train_step

    train_x = list(range(len(tl)))
    eval_x = [i * ratio for i in range(len(el))]

    tl_smooth = pd.Series(tl).rolling(5).mean()

    plt.figure(figsize=(10, 5))

    plt.plot(train_x, tl_smooth, linewidth=1, alpha=0.9, label="train_loss")

    plt.plot(
        eval_x,
        el,
        marker="o",
        markersize=3,
        linewidth=1,
        label="eval_loss"
    )

    plt.xlabel("Step (aligned)")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    # plt.show()
    plt.tight_layout()

    plt.savefig(save_path, dpi=200)
    plt.close()


os.makedirs(IMAGES_DIR, exist_ok=True)

plot_single_log(
    LOG_PRETRAIN_DIR,
    "Pretrain Loss",
    os.path.join(IMAGES_DIR, "pretrain_loss.png")
)

plot_single_log(
    LOG_SFT_DIR,
    "SFT Loss",
    os.path.join(IMAGES_DIR, "sft_loss.png"),
    train_step=20, eval_step=200
)

BENCH_MARK

In [5]:
import json
import re
from collections import defaultdict

def extract_choice(text):
    if not text:
        return None

    match = re.search(
        r"(?:The correct answer is|Đáp án đúng là)\s*([A-Da-d]|[1-5])",
        text
    )

    if not match:
        return None

    val = match.group(1).upper()
    if val in ["A", "B", "C", "D"]:
        return ord(val) - ord("A") + 1
    return int(val)


with open((RESULT_MCQ_DIR).resolve(), "r", encoding="utf-8") as f:
    data = json.load(f)

total = 0
correct = 0
extracted = 0

cat_stats = defaultdict(lambda: {"total": 0, "correct": 0})
wrong_samples = []

for idx, sample in enumerate(data):
    gt = sample.get("answer")
    text = sample.get("model_output") or sample.get("think", "")
    pred = extract_choice(text)

    total += 1
    cat = sample.get("category") or "UNKNOWN"
    cat_stats[cat]["total"] += 1

    if pred is None:
        continue
    else:
        extracted += 1
        if pred == gt:
            correct += 1
            cat_stats[cat]["correct"] += 1
        else:
            wrong_samples.append(sample)

print("\n=== OVERALL ===")
if total > 0:
    print(f"Accuracy: {correct}/{total} = {correct/total:.4f}")
else:
    print("No data")

print("\n=== BY CATEGORY ===")
for cat, stat in cat_stats.items():
    t = stat["total"]
    c = stat["correct"]
    acc = c / t if t > 0 else 0
    print(f"{cat}: {c}/{t} = {acc:.4f}")

print(f"\nTotal wrong predictions: {len(wrong_samples)}")

if wrong_samples:
    with open("wrong_samples.json", "w", encoding="utf-8") as fout:
        json.dump(wrong_samples, fout, ensure_ascii=False, indent=2)
    print(f"Saved {len(wrong_samples)} wrong samples to wrong_samples.json")


=== OVERALL ===
Accuracy: 6928/10000 = 0.6928

=== BY CATEGORY ===
Research publications: 3254/4499 = 0.7233
Standards specifications: 1157/1999 = 0.5788
Research overview: 1438/2000 = 0.7190
Lexicon: 411/500 = 0.8220
Standards overview: 668/1002 = 0.6667

Total wrong predictions: 3072
Saved 3072 wrong samples to wrong_samples.json


In [2]:
import json
from collections import defaultdict
from rouge_score import rouge_scorer
import pandas as pd

def evaluate_rouge_by_type(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True
    )

    scores = defaultdict(lambda: {"r1": [], "r2": [], "rl": []})

    for item in data:
        pred = item.get("model_output", "")
        ref = item.get("answer", "")
        t = item.get("type", "unknown")

        if not pred or not ref:
            continue

        s = scorer.score(ref, pred)

        scores[t]["r1"].append(s["rouge1"].fmeasure)
        scores[t]["r2"].append(s["rouge2"].fmeasure)
        scores[t]["rl"].append(s["rougeL"].fmeasure)

    rows = []
    for t, v in scores.items():
        rows.append({
            "type": t,
            "count": len(v["r1"]),
            "rouge1": sum(v["r1"]) / len(v["r1"]),
            "rouge2": sum(v["r2"]) / len(v["r2"]),
            "rougeL": sum(v["rl"]) / len(v["rl"]),
        })

    df = pd.DataFrame(rows).sort_values("type")
    return df

In [3]:
df = evaluate_rouge_by_type(RESULT_OQNA_DIR)
df

,type,count,rouge1,rouge2,rougeL
0,arxiv,3000,0.906454,0.875511,0.897277
1,standard,6000,0.915462,0.894141,0.908792
2,wiki,1000,0.871730,0.803784,0.865450
